# İP3 — Bloom Taksonomisi NLP Analizi
**Sorumlu:** Sudenaz Güven | **Tarih:** 19 Haziran 2026

Bu notebook `master_clean.xlsx` üzerindeki her öğrenme kazanımını Bloom taksonomisi sözlüğüyle sınıflandırır, ders bazında bilişsel düzey profili çıkarır ve `K_bilissel` katsayısını hesaplar.  
Çıktı: `data/processed/master_bloom.xlsx`

### Kullanılan Kütüphaneler ve Neden Kullanıldıkları

| Kütüphane | Kullanım Amacı |
|-----------|----------------|
| `pandas`  | Veri okuma, sütun ekleme, gruplama ve kaydetme |
| `numpy`   | Sayısal işlemler |
| `openpyxl`| `.xlsx` dosyalarını okumak/yazmak için pandas backend |

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

IN_FILE = Path("../data/processed/master_clean.xlsx")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Bloom sözlüğü: (seviye, ad, ağırlık, anahtar_fiiller) ───────────────
# L6→L1 sırasıyla tanımlanır: ilk eşleşen kazanır (yüksek seviye öncelikli)
BLOOM = [
    (6, "Yaratma",        1.50, ["tasarlar","geliştirir","oluşturur","üretir","planlar","inşa eder","kurar","formüle eder"]),
    (5, "Değerlendirme",  1.40, ["değerlendirir","yargılar","savunur","eleştirir","karar verir","doğrular","önceliklendirir"]),
    (4, "Analiz",         1.30, ["analiz eder","inceler","test eder","sorgular","kıyaslar","ayırt eder","karşılaştırır","karmaşıklığını"]),
    (3, "Uygulama",       1.20, ["uygular","hesaplar","çözer","kullanır","gerçekleştirir","dener","yapar","işletir","seçer"]),
    (2, "Anlama",         1.10, ["açıklar","özetler","sınıflandırır","yorumlar","karşılaştırır","örnekler","betimler","kavrar","ilişkilendirir","anlar"]),
    (1, "Hatırlama",      1.00, ["tanımlar","listeler","adlandırır","belirtir","ifade eder","öğrenir","sıralar","hatırlar","tekrarlar"]),
]

KAZ_COLS = [f"OgrenmeKazanim_{i}" for i in range(1, 26)]

print("Bloom sözlüğü tanımlandı.")
print(f"  Toplam seviye   : {len(BLOOM)}")
print(f"  Toplam fiil     : {sum(len(kw) for _,_,_,kw in BLOOM)}")

Bloom sözlüğü tanımlandı.
  Toplam seviye   : 6
  Toplam fiil     : 51


### Adım 1 — Veri Yükleme

`master_clean.xlsx` dosyası yüklenir.  
Bu dosya İP2'de üretilmiş; 4.133 ders, temizlenmiş ve öznitelik eklenmiş hâlde gelir.

In [2]:
print(f"Girdi yükleniyor: {IN_FILE}")
df = pd.read_excel(IN_FILE)
print(f"  {len(df)} satır, {len(df.columns)} sütun yüklendi.")
print(f"\nİlk 3 satır (kimlik sütunları):")
print(df[["Katalogid","DersAdi","AKTSKredi","n_ogrenme_kazanim"]].head(3).to_string())

Girdi yükleniyor: ../data/processed/master_clean.xlsx


  4133 satır, 88 sütun yüklendi.

İlk 3 satır (kimlik sütunları):
   Katalogid                                   DersAdi  AKTSKredi  n_ogrenme_kazanim
0     201206           Veri Yapıları ve Algoritmaları           5                  5
1    9907120  Temel Bilgisayar Bilimleri (Sosyal) - UE          4                  6
2    9907119           Temel Bilgisayar Bilimleri (UE)          4                  7


### Adım 2 — Bloom Sınıflandırma Mantığı

Her öğrenme kazanımı metni şu kuralla sınıflandırılır:

1. Metin küçük harfe çevrilir
2. Bloom sözlüğü **L6'dan L1'e** taranır
3. İlk eşleşen anahtar fiilin seviyesi atanır (yüksek seviye öncelikli)
4. Hiçbir fiil eşleşmezse güvenli taban olarak **L1** atanır

**Neden Türkçe için bu yaklaşım?**  
Türkçe SOV (Özne-Nesne-Fiil) dil yapısına sahiptir; fiil cümlenin sonunda yer alır.  
Bu nedenle sözlük tam metne uygulanarak sonundaki eylem fiili yakalanır.

In [3]:
def classify_kazanim(text: str) -> int:
    """Tek bir kazanım metnini Bloom seviyesine (1-6) dönüştürür."""
    if not text or str(text).strip().lower() in ("", "nan"):
        return 0  # boş kazanım
    t = str(text).lower()
    for level, _, _, keywords in BLOOM:
        for kw in keywords:
            if kw in t:
                return level
    return 1  # eşleşme yoksa güvenli taban: L1

# Test
ornekler = [
    "Veri yapılarını karşılaştırır ve analiz eder.",
    "Temel kavramları tanımlar.",
    "Proje tasarlar ve geliştirir.",
    "Algoritma uygular.",
]
print("Sınıflandırma testi:")
for ornek in ornekler:
    lv = classify_kazanim(ornek)
    ad = {d[0]: d[1] for d in BLOOM}.get(lv, "?")
    print(f"  L{lv} ({ad:<14}) ← {ornek}")

Sınıflandırma testi:
  L4 (Analiz        ) ← Veri yapılarını karşılaştırır ve analiz eder.
  L1 (Hatırlama     ) ← Temel kavramları tanımlar.
  L6 (Yaratma       ) ← Proje tasarlar ve geliştirir.
  L3 (Uygulama      ) ← Algoritma uygular.


### Adım 3 — Ders Bazlı Bloom Profili Hesaplama

Her ders için tüm kazanımlar sınıflandırılır ve şu metrikler üretilir:

| Sütun | Açıklama |
|-------|----------|
| `bloom_l1_count` … `bloom_l6_count` | Her Bloom seviyesinden kaç kazanım var |
| `bloom_l1_ratio` … `bloom_l6_ratio` | Oransal dağılım (0–1) |
| `bloom_avg_level` | Kazanımların ağırlıksız ortalaması |
| `bloom_max_level` | En yüksek Bloom seviyesi |
| `bloom_dominant_level` | En sık görülen Bloom seviyesi (mod) |
| `K_bilissel` | Bilişsel zorluk katsayısı (formül aşağıda) |

**K_bilissel Formülü:**

$$K = 1 + \frac{\bar{s} - 1}{5} \times 0.50$$

- $\bar{s}$: kazanımların ortalama Bloom seviyesi (1–6)  
- $K$ aralığı: **[1.00, 1.50]** — L1 tamamen hatırlama dersine 1.00, L6 tamamen yaratma dersine 1.50 atanır

In [4]:
def process_row(row) -> dict:
    """Bir dersin tüm kazanımlarını sınıflandırıp Bloom profilini döndürür."""
    levels = []
    for c in KAZ_COLS:
        val = row.get(c, None)
        if pd.notna(val) and str(val).strip().lower() not in ("", "nan"):
            lv = classify_kazanim(str(val))
            if lv > 0:
                levels.append(lv)

    if not levels:
        levels = [1]  # hiç kazanım yoksa L1

    counts  = {i: levels.count(i) for i in range(1, 7)}
    total   = len(levels)
    ratios  = {i: counts[i] / total for i in range(1, 7)}

    avg_level = sum(levels) / total
    max_level = max(levels)
    dom_level = max(counts, key=counts.get)

    # K_bilissel formülü
    K = 1 + ((avg_level - 1) / 5) * 0.50

    justification = (
        f"Dominant: L{dom_level} | "
        f"Ort: {avg_level:.2f} | "
        f"Maks: L{max_level} | "
        f"K={K:.4f}"
    )

    result = {}
    for i in range(1, 7):
        result[f"bloom_l{i}_count"] = counts[i]
        result[f"bloom_l{i}_ratio"] = round(ratios[i], 4)
    result["bloom_avg_level"]       = round(avg_level, 4)
    result["bloom_max_level"]       = max_level
    result["bloom_dominant_level"]  = dom_level
    result["K_bilissel"]            = round(K, 4)
    result["bloom_justification"]   = justification
    return result

# Örnek: ilk satır
print("Örnek ders Bloom profili:")
ornek = process_row(df.iloc[0])
for k, v in ornek.items():
    print(f"  {k:<25}: {v}")

Örnek ders Bloom profili:
  bloom_l1_count           : 2
  bloom_l1_ratio           : 0.4
  bloom_l2_count           : 0
  bloom_l2_ratio           : 0.0
  bloom_l3_count           : 0
  bloom_l3_ratio           : 0.0
  bloom_l4_count           : 1
  bloom_l4_ratio           : 0.2
  bloom_l5_count           : 0
  bloom_l5_ratio           : 0.0
  bloom_l6_count           : 2
  bloom_l6_ratio           : 0.4
  bloom_avg_level          : 3.6
  bloom_max_level          : 6
  bloom_dominant_level     : 1
  K_bilissel               : 1.26
  bloom_justification      : Dominant: L1 | Ort: 3.60 | Maks: L6 | K=1.2600


### Adım 4 — Tüm Derslere Uygulama

`df.apply()` ile `process_row` fonksiyonu her satıra uygulanır.  
Sonuçlar orijinal DataFrame'e sütun olarak eklenir.

In [5]:
print(f"Bloom analizi başlıyor... ({len(df)} ders)")
bloom_rows = df.apply(process_row, axis=1, result_type="expand")
df = pd.concat([df, bloom_rows], axis=1)
print("Tamamlandı.")
print(f"\nEklenen Bloom sütunları ({len(bloom_rows.columns)} adet):")
print("  ", list(bloom_rows.columns))
print(f"\nK_bilissel — ilk 5 değer:")
print(df[["DersAdi","bloom_avg_level","bloom_dominant_level","K_bilissel"]].head().to_string())

Bloom analizi başlıyor... (4133 ders)


Tamamlandı.



Eklenen Bloom sütunları (17 adet):
   ['bloom_l1_count', 'bloom_l1_ratio', 'bloom_l2_count', 'bloom_l2_ratio', 'bloom_l3_count', 'bloom_l3_ratio', 'bloom_l4_count', 'bloom_l4_ratio', 'bloom_l5_count', 'bloom_l5_ratio', 'bloom_l6_count', 'bloom_l6_ratio', 'bloom_avg_level', 'bloom_max_level', 'bloom_dominant_level', 'K_bilissel', 'bloom_justification']

K_bilissel — ilk 5 değer:


                                      DersAdi  bloom_avg_level  bloom_dominant_level  K_bilissel
0             Veri Yapıları ve Algoritmaları            3.6000                     1      1.2600
1    Temel Bilgisayar Bilimleri (Sosyal) - UE           2.3333                     3      1.1333
2             Temel Bilgisayar Bilimleri (UE)           2.2857                     1      1.1286
3    Temel Bilgi Teknolojileri Kullanımı (UE)           2.5000                     3      1.1500
4  Atatürk İlkeleri ve İnkilap Tarihi II (UE)           3.2500                     4      1.2250


### Sonuç

| | Değer |
|---|---|
| **Girdi** | 4.133 satır (`master_clean.xlsx`) |
| **Eklenen sütun** | 13 Bloom sütunu |
| **K_bilissel aralığı** | [1.00, 1.50] |
| **Dosya** | `data/processed/master_bloom.xlsx` ve `master_bloom.csv` |

Bu dosya modelleme aşamasının (İP4) ana girdisini oluşturur.

In [6]:
df.to_excel(OUT_DIR / "master_bloom.xlsx", index=False)
df.to_csv(OUT_DIR / "master_bloom.csv", index=False, encoding="utf-8-sig")

level_labels = {
    1:"L1 Hatırlama", 2:"L2 Anlama", 3:"L3 Uygulama",
    4:"L4 Analiz", 5:"L5 Değerlendirme", 6:"L6 Yaratma"
}

print("=" * 52)
print("          BLOOM ANALİZİ RAPORU")
print("=" * 52)

print("\n── Dominant Düzey Dağılımı ──────────────────────")
for lv, cnt in df["bloom_dominant_level"].value_counts().sort_index().items():
    print(f"  {level_labels.get(lv, f'L{lv}'):<22}: {cnt:>5}  ({cnt/len(df)*100:.1f}%)")

print("\n── Fakülte Bazlı Ort. Bloom Düzeyi ─────────────")
fak_avg = df.groupby("Fakulte")["bloom_avg_level"].mean().sort_values(ascending=False)
for fak, avg in fak_avg.items():
    print(f"  {fak:<22}: {avg:.3f}")

k = df["K_bilissel"]
print("\n── K_bilissel İstatistikleri ────────────────────")
print(f"  Min      : {k.min():.4f}")
print(f"  Max      : {k.max():.4f}")
print(f"  Ortalama : {k.mean():.4f}")
print(f"  Std      : {k.std():.4f}")

print(f"\n  Kaydedildi → {OUT_DIR / 'master_bloom.xlsx'}")
print(f"  Kaydedildi → {OUT_DIR / 'master_bloom.csv'}")

          BLOOM ANALİZİ RAPORU

── Dominant Düzey Dağılımı ──────────────────────
  L1 Hatırlama          :  2810  (68.0%)
  L2 Anlama             :   579  (14.0%)
  L3 Uygulama           :   491  (11.9%)
  L4 Analiz             :   112  (2.7%)
  L5 Değerlendirme      :    34  (0.8%)
  L6 Yaratma            :   107  (2.6%)

── Fakülte Bazlı Ort. Bloom Düzeyi ─────────────
  Siyasal Bilgiler      : 2.651
  Teknoloji             : 2.337
  Mühendislik           : 2.036
  Eğitim                : 1.893
  Spor Bilimleri        : 1.874
  Fen-Edebiyat          : 1.864
  Güzel Sanatlar        : 1.686

── K_bilissel İstatistikleri ────────────────────
  Min      : 1.0000
  Max      : 1.5000
  Ortalama : 1.0981
  Std      : 0.0954

  Kaydedildi → ../data/processed/master_bloom.xlsx
  Kaydedildi → ../data/processed/master_bloom.csv
